# 01 — Species, zones, and abundances


This notebook focuses on the basic objects that replace common NucNet Tools concepts:

- `Species`: one nuclide, with `Z`, `A`, `N`, mass excess, and optional partition data.
- `Zone`: one thermodynamic/composition state.
- `Network`: a collection of species, reactions, and zones.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)

## Species parsing

The parser accepts names like `h1`, `he4`, `ni56`, and `n`. Some aliases are normalized, for example `alpha -> he4`.

In [ ]:
names = ['n', 'p', 'h1', 'alpha', 'he4', 'c12', 'ni56']
rows = []
for name in names:
    sp = nn.Species.parse(name)
    rows.append(dict(input=name, normalized=sp.name, Z=sp.z, A=sp.a, N=sp.n, latex=sp.latex))
pd.DataFrame(rows)

## Create and inspect a zone

A zone stores abundances and thermodynamic properties. In many NucNet workflows, each zone represents one time, trajectory point, or spatial cell.

In [ ]:
z = nn.Zone(label=('trajectory_1','step_0','0'), properties={'t9':'1.5', 'rho':'1e6'})
z.set_abundance('he4', 0.20)
z.set_abundance('c12', 0.01)
z.set_abundance('o16', 0.002)

for key, value in z.abundances.items():
    print(f'{key:>4s}: Y={value:.5e}')
print('T9 =', z.temperature9())
print('rho =', z.density())
print('sum Y =', z.ysum())
print('Ye =', z.ye())
print('mass fractions =', z.mass_fractions())

## Abundance matrix for many zones

The `Network.abundance_matrix()` method gives a matrix with shape `(n_zones, n_species)`. This is useful for plotting trajectories and comparing output files.

In [ ]:
net = nn.Network()
for name in ['he4','c12','o16']:
    net.add_species(nn.Species.parse(name))
for i in range(5):
    zone = nn.Zone(label=('traj', str(i), '0'), properties={'t9': str(1.0+0.1*i), 'rho': '1e6'})
    zone.set_abundance('he4', 0.25*np.exp(-0.3*i))
    zone.set_abundance('c12', 0.005*i)
    zone.set_abundance('o16', 0.001*i*i)
    net.add_zone(zone)

species, Y = net.abundance_matrix(['he4','c12','o16'])
pd.DataFrame(Y, columns=species)

In [ ]:
for j, name in enumerate(species):
    plt.plot(range(Y.shape[0]), Y[:, j], marker='o', label=name)
plt.xlabel('zone index')
plt.ylabel('Y')
plt.yscale('log')
plt.legend();